# Lab 4.1 &mdash; The Tool Contract

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Build the descriptor a model actually receives, and see what it leaves behind
- Decide which failures are worth retrying &mdash; and which never are
- Turn a tool that raises into a tool that returns something the agent can act on
- Tell apart the four kinds of failure that all look like &lsquo;no result&rsquo;

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Start here.** Everything else in Module 4 &mdash; selection accuracy, multi-tool
> orchestration, MCP &mdash; is this contract, either written by you or by someone else.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 4 labs -- the same payment exceptions as Day 1,
# now reached through tools the agent chooses, and then through tools it did not write.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# The tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

## Concept

A tool is not the function you wrote. From the model's side a tool is exactly three fields:

| field | comes from | what it decides |
|---|---|---|
| `name` | the function name | how the tool is referred to |
| `description` | the docstring | **whether it is chosen at all** |
| `parameters` | the signature | whether the arguments are well formed |

The body, the tests and the types you were careful about never cross the boundary. That is the
whole reason a tool-calling bug is usually a writing bug.

## Section 1 &mdash; What actually crosses the boundary

Build the descriptor. Note what you *cannot* put in it.

In [ ]:
import inspect

_JSON_TYPES = {str: "string", int: "integer", float: "number", bool: "boolean"}

def _json_type(annotation) -> str:
    """Map a Python annotation onto a JSON-Schema type name."""
    return _JSON_TYPES.get(annotation, "string")


def tool_descriptor(fn) -> dict:
    """Build the three fields a model receives for one Python function.

    A parameter with no default is required; one with a default is optional.
    """
    props, required = {}, []
    for pname, p in inspect.signature(fn).parameters.items():
        props[pname] = {"type": _json_type(p.annotation)}
        if p.default is inspect.Parameter.empty:
            required.append(pname)
    return {
        "name": fn.__name__,
        # the docstring, whole and verbatim -- including the boundary sentence
        "description": inspect.getdoc(fn) or "",
        "parameters": {"type": "object", "properties": props, "required": required},
    }

In [ ]:
# --- Self-check: Section 1
check("the descriptor has exactly the three fields that cross the boundary",
      lambda: set(tool_descriptor(lookup_payment)) == {"name", "description", "parameters"})
check("the name is the function's own name",
      lambda: tool_descriptor(lookup_payment)["name"] == "lookup_payment")
check("the description is the whole docstring, not just its first line",
      lambda: "Not for searching" in tool_descriptor(lookup_payment)["description"],
      "the boundary sentence is the part that stops the wrong call -- do not truncate it")
check("a parameter with no default is required",
      lambda: tool_descriptor(lookup_payment)["parameters"]["required"] == ["ref"])
check("the implementation does not cross the boundary",
      lambda: "LEDGER" not in json.dumps(tool_descriptor(lookup_payment)),
      "the model never sees the body -- if it did, this check would be meaningless")

guard(lambda: print(json.dumps(tool_descriptor(lookup_payment), indent=2)[:420]))

## Section 2 &mdash; Which failures deserve a retry

Module 2 called blind retry the most common production failure. The fix starts in the tool: a
result that says *whether trying again could possibly help*.

Retrying a malformed argument produces the same malformed argument. Retrying a permission denial
produces the same denial. Only a **transient** failure has earned a second attempt.

In [ ]:
ERROR_KINDS = ("not_found", "invalid_input", "unavailable", "timeout", "not_permitted")

def is_retryable(kind: str) -> bool:
    """True only for failures where the identical call might succeed on a second attempt."""
    return kind in {"unavailable", "timeout"}


def ok(data) -> dict:
    """A successful result."""
    return {"ok": True, "data": data}


def fail(kind: str, message: str) -> dict:
    """A failure the agent can read, route on, and explain -- rather than an exception it cannot."""
    return {"ok": False, "error": kind, "message": message, "retryable": is_retryable(kind)}

In [ ]:
# --- Self-check: Section 2
check("a service that did not answer is worth another try",
      lambda: is_retryable("unavailable") is True)
check("so is a timeout", lambda: is_retryable("timeout") is True)
check("a bad argument is not -- the retry sends the same bad argument",
      lambda: is_retryable("invalid_input") is False)
check("a missing record is not -- it will still be missing",
      lambda: is_retryable("not_found") is False)
check("a permission denial is not -- and retrying it is how you page a security team",
      lambda: is_retryable("not_permitted") is False)
check("every failure carries all four fields",
      lambda: set(fail("not_found", "x")) == {"ok", "error", "message", "retryable"})

## Section 3 &mdash; A tool that never raises

The same lookup, rewritten to the contract. Watch the distinction in the middle: **&ldquo;I looked and
it is not there&rdquo; is not the same fact as &ldquo;I could not look&rdquo;** &mdash; and an agent that
confuses the two reports a missing payment when the ledger was merely down.

In [ ]:
def safe_lookup(ref, ledger_down: bool = False) -> dict:
    """Return one payment to the tool contract. Never raises, whatever it is handed."""
    if not isinstance(ref, str) or not ref.startswith("PMT-"):
        return fail("invalid_input", f"{ref!r} is not a payment reference; expected e.g. 'PMT-1002'")
    if ledger_down:
        return fail("unavailable", "the ledger service did not respond")
    record = LEDGER.get(ref)
    if record is None:
        return fail("not_found", f"no payment on file with reference {ref!r}")
    return ok({"ref": ref, **record})

In [ ]:
# --- Self-check: Section 3
check("a known payment comes back as a success",
      lambda: safe_lookup("PMT-1002")["ok"] is True)
check("and carries the record",
      lambda: safe_lookup("PMT-1002")["data"]["reason_code"] == "INSUFFICIENT_FUNDS")
check("a malformed reference is invalid_input, not not_found",
      lambda: safe_lookup("northwind")["error"] == "invalid_input")
check("a well-formed reference that is absent is not_found",
      lambda: safe_lookup("PMT-9999")["error"] == "not_found")
check("a down ledger is unavailable -- and is the only one of the three worth retrying",
      lambda: safe_lookup("PMT-1002", ledger_down=True)["error"] == "unavailable"
              and safe_lookup("PMT-1002", ledger_down=True)["retryable"] is True)
check("'not there' and 'could not look' are different answers",
      lambda: safe_lookup("PMT-9999")["error"] != safe_lookup("PMT-9999", ledger_down=True)["error"],
      "if these ever collapse into one, the agent will report a payment missing when the ledger blinked")
check("it never raises, whatever it is handed",
      lambda: all(isinstance(safe_lookup(x), dict) for x in (None, 42, "", [], "PMT-1002")))

for probe in ("PMT-1002", "PMT-9999", "northwind"):
    guard(lambda p=probe: print(f"  {p:12} -> {json.dumps(safe_lookup(p))[:96]}"))
guard(lambda: print(f"  {'ledger down':12} -> {json.dumps(safe_lookup('PMT-1002', ledger_down=True))}"))

## Run it for real

Hand the model the descriptor you built &mdash; and nothing else &mdash; and ask it what the tool is for
and when it should *not* be used. If it answers well, your description is doing its job. If it
hedges, the model would have hedged when choosing, too.

In [ ]:
if llm_ready():
    def _probe():
        d = tool_descriptor(lookup_payment)
        return ask(
            "Here is a tool available to an agent, in the exact form the agent receives it.\n\n"
            + json.dumps(d, indent=2)
            + "\n\nIn two sentences: what is this tool for, and when should it NOT be used?")
    out = guard(_probe)
    if out:
        print(out.strip()[:500])

### Read it

The model has no more information than you gave it. Anything it gets wrong here, it would also
get wrong while choosing between four tools under time pressure &mdash; except that there you would
never see it reason about it.

In [ ]:
score()

## Your turn

1. `_json_type` collapses every unknown annotation to `"string"`. Give `search_payments` an
   `Optional[str]` and watch the schema lie about it. What does a lying schema cost you?
2. Add a `not_permitted` path to `safe_lookup` for a reference outside an allowed range. Which of
   the four failure kinds should an agent be allowed to *report to the user verbatim*, and which
   should it summarise?
3. Write the boundary sentence for `release_payment` &mdash; the tool that moves money. Then ask
   yourself whether a sentence is the right control for it. Lab 4.5 says it is not.